In [0]:
#spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_calendario")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_calendario (
  id_calendario LONG,
  fecha DATE,
  ano LONG,
  trimestre LONG,
  mes LONG,
  nombre_mes STRING,
  dia LONG,
  nombre_dia STRING,
  semana_ano LONG
)
""")

In [0]:
import pandas as pd

df_spark = spark.table("workspace.silver.adidas_US_sales")

df = df_spark.toPandas()

#Se genera una serie de fechas desde la fecha mínima a la máxima de la tabla ventas
min_date = df["fecha_factura"].min()
max_date = df["fecha_factura"].max()
date_range = pd.date_range(start=min_date, end=max_date, freq="D")
dim_calendario = pd.DataFrame({"fecha": date_range})

#Diccionarios para nombres en español
dias = {
    'Monday': 'Lunes',
    'Tuesday': 'Martes',
    'Wednesday': 'Miércoles',
    'Thursday': 'Jueves',
    'Friday': 'Viernes',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

meses = {
    'January': 'Enero',
    'February': 'Febrero',
    'March': 'Marzo',
    'April': 'Abril',
    'May': 'Mayo',
    'June': 'Junio',
    'July': 'Julio',
    'August': 'Agosto',
    'September': 'Septiembre',
    'October': 'Octubre',
    'November': 'Noviembre',
    'December': 'Diciembre'
}

#Se crea el df con las fechas a partir de la serie anterior, y llevandola a una columna llamada fecha
dim_calendario["id_calendario"] = dim_calendario["fecha"].dt.strftime("%Y%m%d").astype("int64")
dim_calendario["ano"] = dim_calendario["fecha"].dt.year.astype("int64")
dim_calendario["mes"] = dim_calendario["fecha"].dt.month.astype("int64")
dim_calendario["dia"] = dim_calendario["fecha"].dt.day.astype("int64")
dim_calendario["nombre_dia"] = dim_calendario["fecha"].dt.day_name().map(dias)
dim_calendario["semana_ano"] = dim_calendario["fecha"].dt.isocalendar().week.astype("int64")
dim_calendario["nombre_mes"] = dim_calendario["fecha"].dt.month_name().map(meses)
dim_calendario["trimestre"] = dim_calendario["fecha"].dt.quarter.astype("int64")
dim_calendario["fecha"] = dim_calendario["fecha"].dt.date

#Se ordenan las columnas
dim_calendario = dim_calendario[["id_calendario", "fecha", "ano", "trimestre", "mes", "nombre_mes", "dia", "nombre_dia", "semana_ano"]]

#Se lleva a spark
df_spark = spark.createDataFrame(dim_calendario)

In [0]:
# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_calendario")

In [0]:
%sql
--select * from workspace.gold.dim_calendario LIMIT 10